# Fase 2: Exploratory Data Analysis EPL

Notebook ini mengecek distribusi hasil pertandingan, home advantage, dan perubahan pola per musim untuk EPL 2021/22 sampai 2025/26. Analisis bersifat deskriptif; uji statistik dipakai untuk membedakan pola visual dari bukti yang cukup kuat.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from scipy.stats import binomtest, chi2_contingency, ttest_1samp

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'matches_clean.csv'
matches = pd.read_csv(DATA_PATH, parse_dates=['datetime', 'date'])

RESULT_ORDER = ['H', 'D', 'A']
RESULT_LABELS = {'H': 'Home Win', 'D': 'Draw', 'A': 'Away Win'}
RESULT_COLORS = {'H': '#186F65', 'D': '#D9A441', 'A': '#A33D3D'}

sns.set_theme(style='whitegrid', context='notebook')
print(f'{len(matches):,} pertandingan | {matches.season.nunique()} musim')
print(f'{matches.date.min().date()} sampai {matches.date.max().date()}')

## 1. Distribusi hasil pertandingan

In [ ]:
result_counts = matches['result'].value_counts().reindex(RESULT_ORDER)
result_distribution = pd.DataFrame({
    'result': [RESULT_LABELS[result] for result in RESULT_ORDER],
    'count': result_counts.values,
    'percentage': (result_counts.values / len(matches) * 100),
})

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(
    result_distribution['result'],
    result_distribution['percentage'],
    color=[RESULT_COLORS[result] for result in RESULT_ORDER],
    width=0.62,
)
ax.bar_label(
    bars,
    labels=[
        f"{row.percentage:.2f}%\n({row.count} laga)"
        for row in result_distribution.itertuples()
    ],
    padding=4,
)
ax.set(title='Distribusi Hasil EPL, 2021/22-2025/26', ylabel='Persentase pertandingan', xlabel='')
ax.set_ylim(0, result_distribution['percentage'].max() + 8)
sns.despine(ax=ax)
plt.tight_layout()
plt.show()

result_distribution.round({'percentage': 2})

**Insight:** Home Win adalah hasil paling umum (44,16%), disusul Away Win (31,95%) dan Draw (23,89%). Artinya kelas target tidak seimbang. Model yang sering menebak Home Win bisa terlihat lumayan hanya dari accuracy, sehingga evaluasi nanti wajib melihat performa tiap kelas, terutama Draw. Angka Home Win juga dekat dengan sanity check historis EPL sekitar 45% di scope project.

## 2. Home advantage

Home advantage dilihat dari dua sisi: selisih frekuensi menang kandang vs menang tandang, dan selisih rata-rata gol. Exact binomial test menguji apakah Home Win lebih sering daripada Away Win di antara pertandingan yang menghasilkan pemenang. Draw dikeluarkan hanya dari uji ini karena tidak mendukung sisi kandang maupun tandang.

In [ ]:
home_wins = int((matches['result'] == 'H').sum())
away_wins = int((matches['result'] == 'A').sum())
decisive_matches = home_wins + away_wins
win_test = binomtest(home_wins, decisive_matches, p=0.5, alternative='greater')

goal_difference = matches['goals_home'] - matches['goals_away']
goal_test = ttest_1samp(goal_difference, popmean=0, alternative='greater')
home_rate = home_wins / len(matches) * 100
away_rate = away_wins / len(matches) * 100
mean_home_goals = matches['goals_home'].mean()
mean_away_goals = matches['goals_away'].mean()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
win_bars = axes[0].bar(
    ['Home Win', 'Away Win'],
    [home_rate, away_rate],
    color=[RESULT_COLORS['H'], RESULT_COLORS['A']],
    width=0.58,
)
axes[0].bar_label(win_bars, labels=[f'{home_rate:.2f}%', f'{away_rate:.2f}%'], padding=4)
axes[0].set(title='Win Rate Kandang vs Tandang', ylabel='Persentase semua pertandingan', ylim=(0, 52))

goal_bars = axes[1].bar(
    ['Gol tim kandang', 'Gol tim tandang'],
    [mean_home_goals, mean_away_goals],
    color=[RESULT_COLORS['H'], RESULT_COLORS['A']],
    width=0.58,
)
axes[1].bar_label(goal_bars, labels=[f'{mean_home_goals:.2f}', f'{mean_away_goals:.2f}'], padding=4)
axes[1].set(title='Rata-rata Gol per Pertandingan', ylabel='Rata-rata gol', ylim=(0, 1.9))

for ax in axes:
    sns.despine(ax=ax)
plt.tight_layout()
plt.show()

print(f'Gap win rate       : {home_rate - away_rate:.2f} percentage points')
print(f'Exact binomial test: p = {win_test.pvalue:.3e}')
print(f'Gap rata-rata gol  : {goal_difference.mean():.3f} gol per laga')
print(f'One-sample t-test  : p = {goal_test.pvalue:.3e}')

**Insight:** Home Win unggul 12,21 percentage points atas Away Win. Tim kandang juga mencetak sekitar 0,27 gol lebih banyak per laga (1,60 vs 1,33). Kedua uji menghasilkan `p < 0,001`, jadi gap agregat ini sangat kecil kemungkinannya muncul hanya karena variasi acak jika sebenarnya tidak ada keunggulan kandang. Ini bukti statistik adanya home advantage pada dataset, tetapi bukan bukti kausal tentang penyebabnya.

## 3. Tren per musim

Grafik pertama membandingkan komposisi hasil. Grafik kedua memperlihatkan rata-rata selisih gol kandang minus tandang sebagai ukuran kekuatan home advantage per musim. Chi-square test mengecek apakah distribusi H/D/A secara keseluruhan berhubungan dengan musim.

In [ ]:
season_counts = pd.crosstab(matches['season'], matches['result']).reindex(columns=RESULT_ORDER)
season_percentages = season_counts.div(season_counts.sum(axis=1), axis=0) * 100
season_goal_gap = (
    matches.assign(goal_gap=matches['goals_home'] - matches['goals_away'])
    .groupby('season')['goal_gap']
    .mean()
)
chi2_stat, chi2_pvalue, chi2_dof, _ = chi2_contingency(season_counts)

fig, axes = plt.subplots(2, 1, figsize=(11, 10), gridspec_kw={'height_ratios': [1.2, 1]})
for result in RESULT_ORDER:
    axes[0].plot(
        season_percentages.index,
        season_percentages[result],
        marker='o',
        linewidth=2.4,
        label=RESULT_LABELS[result],
        color=RESULT_COLORS[result],
    )
axes[0].set(title='Distribusi Hasil per Musim', ylabel='Persentase pertandingan', xlabel='')
axes[0].legend(frameon=False, ncol=3)

gap_colors = [RESULT_COLORS['H'] if value >= 0 else RESULT_COLORS['A'] for value in season_goal_gap]
gap_bars = axes[1].bar(season_goal_gap.index, season_goal_gap.values, color=gap_colors, width=0.64)
axes[1].bar_label(gap_bars, labels=[f'{value:+.2f}' for value in season_goal_gap], padding=4)
axes[1].axhline(0, color='#333333', linewidth=1)
axes[1].set(title='Home Advantage dari Selisih Gol per Musim', ylabel='Rata-rata gol kandang - tandang', xlabel='Musim')
axes[1].set_ylim(min(-0.05, season_goal_gap.min() - 0.1), season_goal_gap.max() + 0.13)

for ax in axes:
    sns.despine(ax=ax)
plt.tight_layout()
plt.show()

print(f'Chi-square test: chi2 = {chi2_stat:.3f}, df = {chi2_dof}, p = {chi2_pvalue:.3f}')
season_percentages.rename(columns=RESULT_LABELS).round(2)

**Insight:** Ada fluktuasi visual: Home Win tertinggi pada 2022/23 (48,42%) dan terendah pada 2024/25 (40,79%). Musim 2024/25 sekaligus punya gap gol kandang terkecil (+0,09), sedangkan 2022/23 terbesar (+0,42). Draw naik hingga 27,37% pada 2025/26. Namun chi-square test memberi `p = 0,310`, sehingga dari lima musim ini belum ada bukti statistik yang cukup bahwa distribusi H/D/A berubah secara struktural antar-musim. Kesimpulan yang aman: home advantage selalu positif, tetapi kekuatannya berfluktuasi.

## Ringkasan Fase 2

1. Home Win adalah kelas mayoritas, sementara Draw paling jarang.
2. Home advantage terlihat secara praktis dan signifikan secara statistik pada data agregat.
3. Besarnya home advantage berubah antar-musim, tetapi perubahan distribusi hasil belum signifikan secara keseluruhan.
4. EDA ini belum membuat fitur atau memakai informasi masa depan untuk training model.